# <span style="color:#ff8c00">01 · Exploratory data analysis — VQA v2</span>

**Goal.** Look at the data before modelling: sizes, answer/question types, how well a fixed answer vocabulary covers the
answers (this quantifies what the legacy pipeline, the original code kept in `legacy/`, silently dropped), annotator agreement, question lengths, simple
language-prior baselines scored with the **official metric**, and a look at real images.

Everything here reads the split tables in `data/processed/tables/` (built once from the raw JSON files). Nothing in this notebook trains a model.

# <span style="color:#ff8c00">Setup</span>

In [ ]:
import collections, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from vqa_lab.config import load_paths
from vqa_lab.data import load_tables
from vqa_lab.image import image_path
from vqa_lab.metrics import evaluate, score_question

pd.set_option("display.width", 140, "display.max_colwidth", 60)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})
paths = load_paths()
summary = {}   # numbers collected here are saved at the end

# <span style="color:#ff8c00">1 · Load the four question files and join the annotations by `question_id`</span>

In [ ]:
tables = load_tables(paths)
train, val, test_dev, test = tables["train"], tables["val"], tables["test_dev"], tables["test"]
expected = {"train": 443_757, "val": 214_354, "test": 447_793}
sizes = pd.DataFrame({"questions": [len(train), len(val), len(test_dev), len(test)],
                      "images": [d.image_id.nunique() for d in (train, val, test_dev, test)]},
                     index=["train2014", "val2014", "test-dev2015", "test2015"])
sizes["questions per image"] = (sizes.questions / sizes.images).round(2)
assert (len(train), len(val), len(test)) == tuple(expected.values())
summary["sizes"] = sizes.to_dict()
sizes

Train and val are the two labelled splits; **test-dev** is a labelled-by-server subset of test (scored only through
EvalAI). The counts match the official numbers exactly.

# <span style="color:#ff8c00">2 · Answer types and question types</span>

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
at = pd.concat({"train": train.answer_type.value_counts(normalize=True), "val": val.answer_type.value_counts(normalize=True)}, axis=1) * 100
at.plot.bar(ax=ax[0], rot=0); ax[0].set_title("Answer type share (%)"); ax[0].set_ylabel("%")
qt = train.question_type.value_counts(normalize=True).head(25) * 100
qt[::-1].plot.barh(ax=ax[1]); ax[1].set_title(f"Top-25 question types of {train.question_type.nunique()} (train, %)")
plt.tight_layout(); plt.show()
summary["answer_type_share_val_%"] = (at["val"].round(2)).to_dict()
at.round(2)

# <span style="color:#ff8c00">3 · The answers: what do people say, and how concentrated is it?</span>

In [ ]:
top = train.majority_answer.value_counts()
display(pd.DataFrame({"count": top.head(20), "share %": (top.head(20) / len(train) * 100).round(2)}).T)
print(f"distinct majority answers in train: {len(top):,};  'yes'+'no' alone cover {(top['yes'] + top['no']) / len(train):.1%} of train questions")

# <span style="color:#ff8c00">Vocabulary coverage — what a fixed top-K answer classifier can never get right</span>

The legacy code kept only the **1,000 most frequent answers** and *also removed the other questions from its test set*.
Here we measure, on the official **val** set, how many questions a top-K vocabulary leaves unanswerable.

In [ ]:
ks = [10, 50, 100, 300, 500, 1000, 2000, 3000, 5000]
rank = {a: i for i, a in enumerate(top.index)}
maj_rank = val.majority_answer.map(lambda a: rank.get(a, 10**9))
any_rank = val.answers_raw.map(lambda xs: min(rank.get(a, 10**9) for a in xs))
cov = pd.DataFrame({"K": ks,
    "majority answer in top-K (%)": [(maj_rank < k).mean() * 100 for k in ks],
    "any human answer in top-K (%)": [(any_rank < k).mean() * 100 for k in ks]}).set_index("K").round(2)
fig, ax = plt.subplots(figsize=(7, 4)); cov.plot(ax=ax, marker="o", logx=True); ax.axvline(1000, color="r", ls="--", label="legacy K=1000")
ax.set_ylabel("% of val questions"); ax.legend(); plt.show()
dropped = 100 - cov.loc[1000, "majority answer in top-K (%)"]
print(f"With K=1000 the legacy pipeline dropped {dropped:.1f}% of val questions from its test set (they cannot be answered by that vocabulary).")
summary["val_questions_with_majority_outside_top1000_%"] = round(dropped, 2); summary["coverage"] = cov.reset_index().to_dict("list")
cov

# <span style="color:#ff8c00">Annotator agreement — how noisy is the ground truth?</span>

In [ ]:
def agreement(xs): return collections.Counter(xs).most_common(1)[0][1]
val_agree = val.answers_raw.map(agreement)
distinct = val.answers_raw.map(lambda xs: len(set(xs)))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
val_agree.value_counts().sort_index().plot.bar(ax=ax[0], rot=0); ax[0].set_title("Annotators agreeing with the top answer (of 10)")
distinct.value_counts().sort_index().plot.bar(ax=ax[1], rot=0); ax[1].set_title("Distinct answers per question")
plt.tight_layout(); plt.show()
print(f"unanimous (10/10): {(val_agree == 10).mean():.1%} | >= 6/10 agree: {(val_agree >= 6).mean():.1%} | no two annotators agree on some answer set (top answer < 3): {(val_agree < 3).mean():.1%}")
summary["val_unanimous_%"] = round((val_agree == 10).mean() * 100, 2)

# <span style="color:#ff8c00">4 · Ceilings and trivial baselines with the **official metric**</span>

These numbers need no image and no learning. They tell us what any model must beat, and — because we compute them with the
verified metric port — they double as a sanity check of the evaluation code.

* **Predict each question's own majority answer** — the best any *single-answer* system could do if it knew the label.
* **Best answer inside a top-1000 vocabulary** — the ceiling of a legacy-style classifier on the *full* val set.
* **"yes" for everything**, and **most frequent answer per question type** (learned on train) — classic language-prior baselines.

In [ ]:
import time
t0 = time.time()
gt = val[["question_id", "question_type", "answer_type", "answers_raw"]]
vocab1000 = set(top.index[:1000])
res = {}
res["oracle: own majority answer"] = evaluate(dict(zip(val.question_id, val.majority_answer)), gt)
best = {q: max((a for a in set(xs) if a in vocab1000), key=lambda a: score_question(a, xs), default="")
        for q, xs in zip(val.question_id, val.answers_raw)}
res["ceiling: best answer in top-1000 vocab"] = evaluate(best, gt)
res["baseline: always 'yes'"] = evaluate({q: "yes" for q in val.question_id}, gt)
prior = train.groupby("question_type").majority_answer.agg(lambda s: s.value_counts().idxmax())
res["baseline: prior per question type"] = evaluate({q: prior.get(t, "yes") for q, t in zip(val.question_id, val.question_type)}, gt)
rows = {k: {"overall": v.overall, **v.per_answer_type} for k, v in res.items()}
table = pd.DataFrame(rows).T[["overall", "yes/no", "number", "other"]].round(2)
summary["baselines_val"] = table.to_dict("index")
print(f"({time.time() - t0:.0f}s)"); table

Reading the table: a model that always says "yes" already scores ~25 %; the question-type prior (the 'language prior' —
guess from the question text alone) gets noticeably higher; and even a perfect top-1000 classifier cannot reach the
single-answer oracle because of the vocabulary gap measured above.

# <span style="color:#ff8c00">5 · Questions: how long are they?</span>

In [ ]:
lens = val.question.str.split().map(len)
pcts = {p: int(np.percentile(lens, p)) for p in (50, 90, 99, 99.9, 100)}
print("words per question, percentiles:", pcts, "| share <= 14 words (legacy max_len):", f"{(lens <= 14).mean():.2%}")
lens.value_counts().sort_index().plot.bar(figsize=(9, 3), rot=0, title="Words per question (val)"); plt.show()
summary["question_length_percentiles"] = pcts

# <span style="color:#ff8c00">6 · Why a random split of question–answer pairs leaks (the legacy protocol)</span>

The legacy code merged train+val and split **question–answer pairs** at random (65/35 and 90/10). Each image carries ~5 questions,
so almost every held-out question's image is also in the training part. VQA v2's official splits are by image.

In [ ]:
allq = pd.concat([train, val], ignore_index=True)
rng = np.random.default_rng(0)
for frac in (0.10, 0.35):
    mask = rng.random(len(allq)) < frac
    train_imgs = set(allq.image_id[~mask])
    share = allq.image_id[mask].isin(train_imgs).mean()
    print(f"random {frac:.0%} hold-out: {share:.1%} of held-out questions are about an image seen in training")
print("official train vs val: shared images =", len(set(train.image_id) & set(val.image_id)))

# <span style="color:#ff8c00">7 · Look at the data</span>

In [ ]:
sample = val.sample(6, random_state=3)
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, (_, r) in zip(axes.ravel(), sample.iterrows()):
    img = Image.open(image_path(paths.raw_dir / "val2014", "val2014", r.image_id))
    ax.imshow(img); ax.axis("off")
    others = val[(val.image_id == r.image_id) & (val.question_id != r.question_id)].head(2)
    lines = [f"Q: {r.question}\nA: {r.majority_answer}  ({collections.Counter(r.answers_raw).most_common(1)[0][1]}/10 agree)"]
    lines += [f"Q: {o.question}\nA: {o.majority_answer}" for o in others.itertuples()]
    ax.set_title("\n".join(lines), fontsize=9, loc="left")
plt.tight_layout(); plt.show()

In [ ]:
sz = [Image.open(image_path(paths.raw_dir / "val2014", "val2014", i)).size for i in val.image_id.drop_duplicates().sample(400, random_state=0)]
w, h = np.array(sz).T
print(f"image size (400 random val images): width {w.min()}–{w.max()} (median {int(np.median(w))}), height {h.min()}–{h.max()} (median {int(np.median(h))})")
print("most common sizes:", collections.Counter(sz).most_common(3))

# <span style="color:#ff8c00">8 · Save the numbers</span>

In [ ]:
out = paths.artifacts_dir; out.mkdir(parents=True, exist_ok=True)
(out / "eda_summary.json").write_text(json.dumps(summary, indent=2, default=float))
print("saved", out / "eda_summary.json")